In [2]:
import pandas as pd

df = pd.read_excel("../data/raw/customer_churn.xlsx")
print(df.shape)
print(df.columns.tolist())

(1500, 33)
['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value', 'Churn Score', 'CLTV', 'Churn Reason']


In [3]:
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,8732-ASKAJ,1,United States,California,San Jose,95171,"34.05, -118.25",37.595214,-119.772546,Male,...,Two Year,Yes,Mailed Check,103.83,3329.4,No,0,26,2910,NaN
1,6818-RWZBW,1,United States,California,Los Angeles,90916,"34.05, -118.25",36.897923,-114.268401,Female,...,Month-to-Month,Yes,Electronic Check,35.55,75.51,Yes,1,36,1817,Network reliability
2,1087-ZUYBB,1,United States,California,San Diego,95313,"34.05, -118.25",39.959017,-121.860737,Female,...,One Year,No,Electronic Check,31.35,1585.56,No,0,4,5318,NaN
3,9868-GKHDB,1,United States,California,Fresno,94414,"34.05, -118.25",37.735553,-120.320586,Male,...,Month-to-Month,Yes,Credit Card (automatic),92.38,3066.56,Yes,1,82,4067,Price too high
4,2218-WAXFQ,1,United States,California,Los Angeles,94713,"34.05, -118.25",41.024972,-114.875972,Male,...,Month-to-Month,Yes,Electronic Check,63.48,3598.15,No,0,15,3901,NaN


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         1500 non-null   str    
 1   Count              1500 non-null   int64  
 2   Country            1500 non-null   str    
 3   State              1500 non-null   str    
 4   City               1500 non-null   str    
 5   Zip Code           1500 non-null   int64  
 6   Lat Long           1500 non-null   str    
 7   Latitude           1500 non-null   float64
 8   Longitude          1500 non-null   float64
 9   Gender             1500 non-null   str    
 10  Senior Citizen     1500 non-null   str    
 11  Partner            1500 non-null   str    
 12  Dependents         1500 non-null   str    
 13  Tenure Months      1500 non-null   int64  
 14  Phone Service      1500 non-null   str    
 15  Multiple Lines     1500 non-null   str    
 16  Internet Service   1500 non-null   

In [5]:
print("Missing values:")
print(df.isnull().sum())

Missing values:
CustomerID              0
Count                   0
Country                 0
State                   0
City                    0
Zip Code                0
Lat Long                0
Latitude                0
Longitude               0
Gender                  0
Senior Citizen          0
Partner                 0
Dependents              0
Tenure Months           0
Phone Service           0
Multiple Lines          0
Internet Service        0
Online Security         0
Online Backup           0
Device Protection       0
Tech Support            0
Streaming TV            0
Streaming Movies        0
Contract                0
Paperless Billing       0
Payment Method          0
Monthly Charges         0
Total Charges           0
Churn Label             0
Churn Value             0
Churn Score             0
CLTV                    0
Churn Reason         1008
dtype: int64


In [6]:
print("Duplicates:", df.duplicated().sum())

Duplicates: 0


In [7]:
print(df["Churn Label"].value_counts())
print(df["Churn Label"].value_counts(normalize=True) * 100)

Churn Label
No     1008
Yes     492
Name: count, dtype: int64
Churn Label
No     67.2
Yes    32.8
Name: proportion, dtype: float64


In [8]:
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce").fillna(0)
df = df.drop_duplicates()
df["Total Charges"].describe()

count    1500.000000
mean     2524.983047
std      1911.869818
min         0.000000
25%       975.115000
50%      2036.145000
75%      3691.225000
max      8533.580000
Name: Total Charges, dtype: float64

Drop columns that are either non-predictive (ID, location detail) or leak the target
- Churn Label duplicates `Churn Value`
- Churn Reason is free text, mostly missing for non-churned customers
- Churn Score is IBM's own churn model output - using it as a feature would be leakage

In [9]:
drop_cols = [
    "CustomerID", "Count", "Country", "State", "City", "Zip Code",
    "Lat Long", "Latitude", "Longitude",
    "Churn Label", "Churn Reason", "Churn Score",
]
df_clean = df.drop(columns=[c for c in drop_cols if c in df.columns])
print(df_clean.shape)
df_clean.head()

(1500, 21)


,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,...,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value,CLTV
0,Male,No,No,No,32,Yes,Yes,Fiber Optic,No,Yes,...,No,No,Yes,Two Year,Yes,Mailed Check,103.83,3329.40,0,2910
1,Female,No,No,No,1,Yes,Yes,Fiber Optic,No,No,...,Yes,No,No,Month-to-Month,Yes,Electronic Check,35.55,75.51,1,1817
2,Female,No,No,Yes,49,Yes,Yes,DSL,Yes,No,...,Yes,No,Yes,One Year,No,Electronic Check,31.35,1585.56,0,5318
3,Male,No,No,No,33,Yes,Yes,Fiber Optic,Yes,No,...,No,Yes,Yes,Month-to-Month,Yes,Credit Card (automatic),92.38,3066.56,1,4067
4,Male,No,Yes,Yes,56,Yes,Yes,Fiber Optic,Yes,Yes,...,Yes,No,No,Month-to-Month,Yes,Electronic Check,63.48,3598.15,0,3901


In [10]:
df_clean.to_csv("../data/processed/churn_clean_raw.csv", index=False)
print("Saved cleaned (not yet encoded) data.")

Saved cleaned (not yet encoded) data.
